# Stage 1: Liveness Detection VAE Pre-training

**Purpose**: Train a Band-Split VAE on real/live data only (one-class learning)

**Dataset**: 20GB processed lip landmark sequences from Google Drive

**Method**: Frequency-Decoupled Feature-Space VAE
- 3-band frequency decomposition (LF/BP/HF)
- Per-band TCN-VAE
- Learnable weighted fusion
- **Float16 precision** for memory efficiency

**Output**: `stage1_pretrained.pt` model checkpoint

## 1. Setup: Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install scipy scikit-learn seaborn tqdm -q

In [ ]:
# Extract dataset from zip file
import os
import zipfile

zip_path = '/content/drive/MyDrive/LRdataset/20GBprocessed.zip'
extract_path = '/content/20GBprocessed'

print(f"Extracting {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print(f"Extraction complete!")
print(f"Dataset path: {extract_path}")

# Check extracted files
npz_files = [f for f in os.listdir(extract_path) if f.endswith('.npz')]
print(f"Found {len(npz_files)} .npz files")

In [ ]:
# Check GPU availability and set float16 as default
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set default dtype to float16 for memory efficiency
torch.set_default_dtype(torch.float16)
print("\n⚠️  Set default dtype to float16 for memory efficiency")
print(f"Current default dtype: {torch.get_default_dtype()}")

In [ ]:
%%javascript
// Keep Colab Runtime Alive by cycling through cells
function KeepClicking(){
    // Get all code cells
    var cells = document.querySelectorAll('div.cell');
    var currentCell = 0;
    
    setInterval(function(){
        // Focus on current cell
        if(cells[currentCell]){
            cells[currentCell].click();
            console.log('Clicked cell ' + currentCell + ' to keep runtime alive');
        }
        
        // Move to next cell (cycle through)
        currentCell = (currentCell + 1) % cells.length;
    }, 60000); // Every 60 seconds
}

KeepClicking();
console.log('Runtime keeper started - will cycle through cells every 60 seconds');

## 2. Model Definition: Band-Split VAE

In [ ]:
# Core imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from scipy import signal
import glob
import time
from tqdm import tqdm

In [ ]:
# Butterworth Filter Bank for frequency decomposition
class ButterworthFilterBank:
    """
    Separates position signal into 3 frequency bands:
    - Low-Frequency (LF): 0 ~ fc_low Hz (global motion)
    - Band-Pass (BP): fc_low ~ fc_high Hz (speech motion)
    - High-Frequency (HF): fc_high ~ Nyquist Hz (micro-motion/artifacts)
    """
    def __init__(self, fps=30, fc_low=2.0, fc_high=8.0, order=4):
        self.fps = fps
        nyquist = fps / 2.0
        
        wn_low = fc_low / nyquist
        wn_high = fc_high / nyquist
        
        # Design Butterworth filters
        self.sos_lf = signal.butter(order, wn_low, btype='lowpass', output='sos')
        self.sos_bp = signal.butter(order, [wn_low, wn_high], btype='bandpass', output='sos')
        self.sos_hf = signal.butter(order, wn_high, btype='highpass', output='sos')
    
    def apply(self, x):
        """Apply filter bank to position landmarks [T, K, 2]"""
        T, K, D = x.shape
        x_flat = x.transpose(1, 2, 0).reshape(K * D, T)
        
        # Apply filters
        x_lf_flat = signal.sosfiltfilt(self.sos_lf, x_flat, axis=1)
        x_bp_flat = signal.sosfiltfilt(self.sos_bp, x_flat, axis=1)
        x_hf_flat = signal.sosfiltfilt(self.sos_hf, x_flat, axis=1)
        
        # Reshape back
        x_lf = x_lf_flat.reshape(K, D, T).transpose(2, 0, 1)
        x_bp = x_bp_flat.reshape(K, D, T).transpose(2, 0, 1)
        x_hf = x_hf_flat.reshape(K, D, T).transpose(2, 0, 1)
        
        return x_lf, x_bp, x_hf

In [ ]:
# TCN Building Block
class TCNBlock(nn.Module):
    """Temporal Convolutional Network Block with residual connection"""
    def __init__(self, C_in, C_out, dilation=1, kernel_size=3):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        
        self.conv = nn.Conv1d(C_in, C_out, kernel_size=kernel_size, 
                              padding=padding, dilation=dilation)
        self.gn = nn.GroupNorm(1, C_out)
        self.act = nn.SiLU()
        self.res = nn.Conv1d(C_in, C_out, 1) if C_in != C_out else nn.Identity()
    
    def forward(self, x):
        y = self.act(self.gn(self.conv(x)))
        return y + self.res(x)

In [ ]:
# Single-Band VAE (for one frequency band)
class SingleBandVAE(nn.Module):
    """Feature-Space VAE for one frequency band"""
    def __init__(self, C_in, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        
        # Encoder
        self.enc_inp = nn.Conv1d(C_in, C_h, 1)
        enc_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in dilations]
        self.encoder = nn.Sequential(*enc_blocks)
        self.enc_out = nn.Conv1d(C_h, C_z * 2, 1)
        
        # Decoder
        self.dec_inp = nn.Conv1d(C_z, C_h, 1)
        dec_blocks = [TCNBlock(C_h, C_h, dilation=d) for d in reversed(dilations)]
        self.decoder = nn.Sequential(*dec_blocks)
        self.dec_out = nn.Conv1d(C_h, C_in, 1)
    
    def encode(self, x):
        h = self.encoder(self.enc_inp(x))
        mu, logvar = torch.chunk(self.enc_out(h), 2, dim=1)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = self.decoder(self.dec_inp(z))
        return self.dec_out(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar

In [ ]:
# Band-Split VAE (3-band architecture)
class BandSplitVAE(nn.Module):
    """Frequency-Decoupled Feature-Space VAE with 3 independent VAEs"""
    def __init__(self, C_in_per_band, C_h=48, C_z=12, dilations=[1, 2, 4]):
        super().__init__()
        
        # 3 independent VAEs
        self.vae_lf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_bp = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        self.vae_hf = SingleBandVAE(C_in_per_band, C_h, C_z, dilations)
        
        # Learnable fusion weights
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3.0)
    
    def forward(self, x_lf, x_bp, x_hf):
        # Forward through each VAE
        x_hat_lf, mu_lf, logvar_lf = self.vae_lf(x_lf)
        x_hat_bp, mu_bp, logvar_bp = self.vae_bp(x_bp)
        x_hat_hf, mu_hf, logvar_hf = self.vae_hf(x_hf)
        
        # Weighted fusion
        weights = F.softmax(self.fusion_weights, dim=0)
        x_hat_fused = (weights[0] * x_hat_lf + 
                       weights[1] * x_hat_bp + 
                       weights[2] * x_hat_hf)
        
        recons = {'lf': x_hat_lf, 'bp': x_hat_bp, 'hf': x_hat_hf}
        mus = {'lf': mu_lf, 'bp': mu_bp, 'hf': mu_hf}
        logvars = {'lf': logvar_lf, 'bp': logvar_bp, 'hf': logvar_hf}
        
        return recons, mus, logvars, x_hat_fused

In [ ]:
# Loss Functions
def kl_divergence(mu, logvar):
    """KL divergence for VAE"""
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    return kl.mean()

def band_split_vae_loss(recons, mus, logvars, targets, 
                        x_hat_fused, x_target_full,
                        betas={'lf': 1.0, 'bp': 1.0, 'hf': 1.0},
                        alpha_fusion=0.0):
    """Band-Split VAE Loss: Per-band reconstruction + KL"""
    loss_dict = {}
    total_loss = 0.0
    
    for band in ['lf', 'bp', 'hf']:
        recon_loss = F.l1_loss(recons[band], targets[band])
        kl_loss = kl_divergence(mus[band], logvars[band])
        band_loss = recon_loss + betas[band] * kl_loss
        
        loss_dict[f'recon_{band}'] = recon_loss.item()
        loss_dict[f'kl_{band}'] = kl_loss.item()
        total_loss += band_loss
    
    # Fusion penalty (optional, not used in Stage 1)
    if alpha_fusion > 0 and x_target_full is not None:
        fusion_loss = F.l1_loss(x_hat_fused, x_target_full)
        loss_dict['fusion'] = fusion_loss.item()
        total_loss += alpha_fusion * fusion_loss
    
    loss_dict['total'] = total_loss.item()
    return total_loss, loss_dict

## 3. Dataset Definition

In [ ]:
# Feature Engineering Functions
def central_diff(arr):
    """Central difference (length-preserving)"""
    d = np.empty_like(arr)
    d[1:-1] = (arr[2:] - arr[:-2]) / 2.0
    d[0] = arr[1] - arr[0]
    d[-1] = arr[-1] - arr[-2]
    return d

def second_diff(arr):
    """Second-order difference (length-preserving)"""
    dd = np.empty_like(arr)
    dd[1:-1] = arr[2:] - 2 * arr[1:-1] + arr[:-2]
    dd[0] = arr[1] - arr[0]
    dd[-1] = arr[-1] - arr[-2]
    return dd

def umeyama_similarity(X, Y):
    """Umeyama similarity transform"""
    Xc = X.mean(axis=0)
    Yc = Y.mean(axis=0)
    X0 = X - Xc
    Y0 = Y - Yc
    
    var = (X0**2).sum() / X0.shape[0] + 1e-12
    U, S, Vt = np.linalg.svd((Y0.T @ X0) / X0.shape[0])
    R = U @ Vt
    
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = U @ Vt
    
    s = S.sum() / var
    t = Yc - s * (R @ Xc)
    return s, R, t

def apply_similarity(P, s, R, t):
    """Apply similarity transform"""
    return (s * (P @ R.T)) + t

def normalize_landmarks_procrustes(landmarks, ref_frames=10):
    """Procrustes normalization"""
    T, K, _ = landmarks.shape
    ref_shape = landmarks[:min(ref_frames, T)].mean(axis=0)
    
    normalized = np.empty_like(landmarks)
    for t in range(T):
        try:
            s, R, tt = umeyama_similarity(landmarks[t], ref_shape)
            normalized[t] = apply_similarity(landmarks[t], s, R, tt)
        except (np.linalg.LinAlgError, ValueError):
            normalized[t] = landmarks[t]
    
    return normalized

def compute_band_features(x_band, fps=30, use_acceleration=False, 
                         use_angle=False, use_angle_rate=False):
    """Compute features for one frequency band"""
    features = {}
    features['position'] = x_band
    features['velocity'] = central_diff(x_band) * fps
    
    if use_acceleration:
        features['acceleration'] = second_diff(x_band) * (fps ** 2)
    
    if use_angle or use_angle_rate:
        v = np.roll(x_band, -1, axis=1) - x_band
        angles = np.arctan2(v[..., 1], v[..., 0])
        angles_unwrap = np.unwrap(angles, axis=0)[..., None]
        
        if use_angle:
            features['angle'] = angles_unwrap
        if use_angle_rate:
            features['angle_rate'] = central_diff(angles_unwrap) * fps
    
    return features

def features_to_array(features):
    """Convert feature dict to array"""
    arrays = []
    for key in ['position', 'velocity', 'acceleration', 'angle', 'angle_rate']:
        if key in features:
            arrays.append(features[key])
    return np.concatenate(arrays, axis=2)

In [ ]:
# Dataset Class with float16 precision
class LipLivenessBandDataset(Dataset):
    """Band-Split VAE Dataset (using float16 for memory efficiency)"""
    def __init__(self, data_dir, T_fixed=300, fps=30, ref_frames=10,
                 use_procrustes=True, use_acceleration=True, 
                 use_angle=True, use_angle_rate=True,
                 fc_low=2.0, fc_high=8.0, filter_order=4, random_crop=False):
        self.data_dir = data_dir
        self.T_fixed = T_fixed
        self.fps = fps
        self.ref_frames = ref_frames
        self.use_procrustes = use_procrustes
        self.use_acceleration = use_acceleration
        self.use_angle = use_angle
        self.use_angle_rate = use_angle_rate
        self.random_crop = random_crop
        
        # Butterworth Filter Bank
        self.filter_bank = ButterworthFilterBank(fps, fc_low, fc_high, filter_order)
        
        # Load file list
        self.files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
        if not self.files:
            raise FileNotFoundError(f"No .npz files found in {data_dir}")
        
        print(f"Found {len(self.files)} samples in {data_dir}")
        print(f"Butterworth Filter Bank: fc_low={fc_low}Hz, fc_high={fc_high}Hz, order={filter_order}")
        print(f"Using float16 precision for memory efficiency")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        # Load .npz file
        data = np.load(self.files[idx])
        lips_outer = data['lips_outer']
        lips_inner = data['lips_inner']
        
        # Concatenate (K=40 landmarks)
        lips = np.concatenate([
            lips_outer[:, :-1, :],
            lips_inner[:, :-1, :]
        ], axis=1)
        
        size = data['size']
        h, w = size
        
        # Normalize by image size
        lips_norm = lips.copy()
        lips_norm[..., 0] /= (w + 1e-8)
        lips_norm[..., 1] /= (h + 1e-8)
        
        # Crop/Pad to T_fixed
        T = lips_norm.shape[0]
        if T > self.T_fixed:
            if self.random_crop:
                start = np.random.randint(0, T - self.T_fixed + 1)
            else:
                start = (T - self.T_fixed) // 2
            lips_norm = lips_norm[start:start + self.T_fixed]
        elif T < self.T_fixed:
            lips_norm = np.pad(lips_norm, ((0, self.T_fixed - T), (0, 0), (0, 0)), mode='edge')
        
        # Geometric normalization
        if self.use_procrustes:
            lips_norm = normalize_landmarks_procrustes(lips_norm, self.ref_frames)
        
        # Frequency decomposition
        lips_lf, lips_bp, lips_hf = self.filter_bank.apply(lips_norm)
        
        # Per-band feature engineering
        features_lf = compute_band_features(lips_lf, self.fps, self.use_acceleration,
                                           self.use_angle, self.use_angle_rate)
        features_bp = compute_band_features(lips_bp, self.fps, self.use_acceleration,
                                           self.use_angle, self.use_angle_rate)
        features_hf = compute_band_features(lips_hf, self.fps, self.use_acceleration,
                                           self.use_angle, self.use_angle_rate)
        
        # Convert to arrays [T, K, F]
        feat_lf = features_to_array(features_lf)
        feat_bp = features_to_array(features_bp)
        feat_hf = features_to_array(features_hf)
        
        # Reshape to [C, T] where C = K × F_dim
        # Use float16 for memory efficiency
        x_lf = torch.tensor(feat_lf, dtype=torch.float16).permute(1, 2, 0).reshape(-1, self.T_fixed)
        x_bp = torch.tensor(feat_bp, dtype=torch.float16).permute(1, 2, 0).reshape(-1, self.T_fixed)
        x_hf = torch.tensor(feat_hf, dtype=torch.float16).permute(1, 2, 0).reshape(-1, self.T_fixed)
        
        # Per-band normalization
        x_lf = (x_lf - x_lf.mean()) / (x_lf.std() + 1e-8)
        x_bp = (x_bp - x_bp.mean()) / (x_bp.std() + 1e-8)
        x_hf = (x_hf - x_hf.mean()) / (x_hf.std() + 1e-8)
        
        return x_lf, x_bp, x_hf

## 4. Training Configuration & Functions

In [ ]:
# Configuration
class Config:
    # Data
    data_dir = '/content/20GBprocessed'
    T_fixed = 300  # 10 seconds @ 30fps
    fps = 30
    
    # Filter Bank
    fc_low = 2.0
    fc_high = 8.0
    filter_order = 4
    
    # Features (FULL version)
    use_acceleration = True
    use_angle = True
    use_angle_rate = True
    K_landmarks = 40
    F_dim = 8  # position + velocity + acceleration + angle + angle_rate
    C_in_per_band = K_landmarks * F_dim  # 320 channels per band
    
    # Model
    C_h = 48
    C_z = 12
    dilations = [1, 2, 4]
    
    # Training
    batch_size = 64
    lr = 1e-3
    epochs = 20
    val_split = 0.1
    num_workers = 4
    
    # Output
    output_dir = '/content/drive/MyDrive/liveness_checkpoints'

config = Config()

# Create output directory
os.makedirs(config.output_dir, exist_ok=True)

print("Configuration:")
print(f"  Data: {config.data_dir}")
print(f"  T_fixed: {config.T_fixed} frames")
print(f"  Features: FULL (8 per landmark)")
print(f"  C_in_per_band: {config.C_in_per_band}")
print(f"  Model: C_h={config.C_h}, C_z={config.C_z}")
print(f"  Training: {config.epochs} epochs, batch_size={config.batch_size}")
print(f"  Precision: float16")
print(f"  Output: {config.output_dir}")

In [ ]:
# Training & Validation Functions
@torch.no_grad()
def validate(model, loader, device):
    """Validate the model"""
    model.eval()
    
    total_loss = 0.0
    lf_rec_loss = 0.0
    bp_rec_loss = 0.0
    hf_rec_loss = 0.0
    n_batches = len(loader)
    
    for x_lf, x_bp, x_hf in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        
        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
        
        loss, loss_dict = band_split_vae_loss(
            recons, mus, logvars, targets,
            x_hat_fused, None, betas=betas, alpha_fusion=0.0
        )
        
        total_loss += loss_dict['total']
        lf_rec_loss += loss_dict['recon_lf']
        bp_rec_loss += loss_dict['recon_bp']
        hf_rec_loss += loss_dict['recon_hf']
    
    return {
        'total': total_loss / n_batches,
        'lf_rec': lf_rec_loss / n_batches,
        'bp_rec': bp_rec_loss / n_batches,
        'hf_rec': hf_rec_loss / n_batches
    }

def train_epoch(model, loader, optimizer, device, use_amp=True):
    """Train for one epoch"""
    model.train()
    
    total_loss = 0.0
    lf_rec_loss = 0.0
    bp_rec_loss = 0.0
    hf_rec_loss = 0.0
    n_batches = len(loader)
    
    scaler = torch.cuda.amp.GradScaler() if use_amp and torch.cuda.is_available() else None
    
    pbar = tqdm(loader, desc="Training")
    for x_lf, x_bp, x_hf in pbar:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)
        
        optimizer.zero_grad()
        
        if use_amp and torch.cuda.is_available():
            with torch.cuda.amp.autocast():
                recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
                targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
                betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
                loss, loss_dict = band_split_vae_loss(
                    recons, mus, logvars, targets,
                    x_hat_fused, None, betas=betas, alpha_fusion=0.0
                )
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
            targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
            betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
            loss, loss_dict = band_split_vae_loss(
                recons, mus, logvars, targets,
                x_hat_fused, None, betas=betas, alpha_fusion=0.0
            )
            loss.backward()
            optimizer.step()
        
        total_loss += loss_dict['total']
        lf_rec_loss += loss_dict['recon_lf']
        bp_rec_loss += loss_dict['recon_bp']
        hf_rec_loss += loss_dict['recon_hf']
        
        pbar.set_postfix({'loss': f"{loss_dict['total']:.4f}"})
    
    return {
        'total': total_loss / n_batches,
        'lf_rec': lf_rec_loss / n_batches,
        'bp_rec': bp_rec_loss / n_batches,
        'hf_rec': hf_rec_loss / n_batches
    }

## 5. Main Training Loop

In [ ]:
# Load Dataset
print("Loading dataset...")
full_dataset = LipLivenessBandDataset(
    data_dir=config.data_dir,
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=True  # Random crop for training
)

# Split into train/val (90/10)
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True
)

print(f"Train size: {train_size}, Val size: {val_size}")

In [ ]:
# Create Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(f"Model dtype: {next(model.parameters()).dtype}")

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=config.lr)

In [ ]:
# Training Loop
print("="*80)
print("Starting training...")
print("="*80)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, config.epochs + 1):
    epoch_start_time = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, device, use_amp=True)
    
    # Validate
    val_loss = validate(model, val_loader, device)
    
    epoch_time = time.time() - epoch_start_time
    
    # Log
    print(
        f"Epoch {epoch}/{config.epochs} - "
        f"train_loss: {train_loss['total']:.4f} "
        f"(lf:{train_loss['lf_rec']:.4f}, bp:{train_loss['bp_rec']:.4f}, hf:{train_loss['hf_rec']:.4f}), "
        f"val_loss: {val_loss['total']:.4f} "
        f"(lf:{val_loss['lf_rec']:.4f}, bp:{val_loss['bp_rec']:.4f}, hf:{val_loss['hf_rec']:.4f}) "
        f"[{epoch_time/60:.1f}min]"
    )
    
    history['train_loss'].append(train_loss['total'])
    history['val_loss'].append(val_loss['total'])
    
    # Save best model
    if val_loss['total'] < best_val_loss:
        best_val_loss = val_loss['total']
        checkpoint_path = os.path.join(config.output_dir, "stage1_pretrained.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
            'config': config
        }, checkpoint_path)
        print(f"  → Saved best model (val_loss={best_val_loss:.4f})")

print("="*80)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")
print(f"Model saved to: {os.path.join(config.output_dir, 'stage1_pretrained.pt')}")
print("="*80)

## 6. Plot Training History

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'], label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Stage 1 Training History (Float16)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(config.output_dir, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

## Done!

**Training complete!** The model checkpoint `stage1_pretrained.pt` has been saved to your Google Drive.

**Configuration**:
- Float16 precision for memory efficiency
- Runtime keeper active (cells cycling every 60s)

**Next steps**:
1. Download the checkpoint from Google Drive
2. Use it for Stage 2 training with real/fake classification